In [ ]:
# ler excel
import pandas as pd

# folha Resumo
df_smh = pd.read_excel('procedimentos.xlsx', sheet_name='Resumo')
# Exibir as primeiras linhas do DataFrame

In [ ]:
df_icd10 = pd.read_excel('icd10cm.xlsx', sheet_name='ICD10PCS')
print(df_icd10.head())

In [ ]:
# filtrar df_id10 coluna Código para corresponder à coluna Diagnóstico em df_smh
filtered_icd10 = df_icd10[df_icd10['Código ICD-10-PCS'].isin(df_smh['Código do procedimento'])]
print(len(filtered_icd10))

In [ ]:

# guardar 'Descrição PT_(Longa)', 'Capitulo ICD-10-CM_desc',      'Descrição PT_(Curta)','Código', 'Válido',
bd = filtered_icd10[['Código ICD-10-PCS', '2ºDigito- Desc PT', '4ºDigito- Desc PT', 'Válido']]

# guardar em excel
print(len(bd))


In [ ]:
#  'Capitulo ICD-10-CM_desc_PT'  unique
capitulos = bd['2ºDigito- Desc PT'].unique()

sections = bd['4ºDigito- Desc PT'].unique()



In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# =======================================
# 1) Conexão com o MySQL usando SQLAlchemy
# =======================================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# =======================================
# 2) Inserir dados da lista/Series sections
# =======================================

with engine.connect() as conn:
    for item in sections:   # se for DataFrame, troque para sections['coluna']
        conn.execute(
            text("INSERT INTO sections (name, kind) VALUES (:name, :kind)"),
            {"name": item, "kind": "procedimentos"}
        )
    conn.commit()
print("Dados inseridos com sucesso!")

In [ ]:
with engine.connect() as conn:
    for item in capitulos:   # se for DataFrame, troque para sections['coluna']
        conn.execute(
            text("INSERT INTO categories (name, kind) VALUES (:name, :kind)"),
            {"name": item, "kind": "diagnosticos"}
        )

    conn.commit()
print("Dados inseridos com sucesso!")

In [20]:
import pandas as pd
from sqlalchemy import create_engine, text

# ==========================
# 1) Criar engine SQLite
# ==========================
engine = create_engine("mysql+pymysql://root:@localhost/codificacao")

# ==========================
# 2) Preparar caches
# ==========================
cache_category = {}
cache_section = {}
cache_codigo = {}
cache_specialty = {}

# ==========================
# 3) Buscar specialty_id apenas uma vez
# ==========================
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT id FROM specialties WHERE name = :name"),
        {"name": "Cirurgia Geral"}
    ).fetchone()
    specialty_id = result[0] if result else None
    cache_specialty['Cirurgia Geral'] = specialty_id
    print(cache_specialty)

# ==========================
# 4) Preparar lista de inserções
# ==========================
registros = []

with engine.connect() as conn:
    for row in bd.itertuples():
        
        # ----- Categoria -----
        nome_categoria = row._2
        if nome_categoria not in cache_category:
            r = conn.execute(
                text("SELECT id FROM categories WHERE name = :name"),
                {"name": nome_categoria}
            ).fetchone()
            cache_category[nome_categoria] = r[0] if r else None

        # ----- Seção -----
        nome_secao = row._3
        if nome_secao not in cache_section:
            r = conn.execute(
                text("SELECT id FROM sections WHERE name = :name"),
                {"name": nome_secao}
            ).fetchone()
            cache_section[nome_secao] = r[0] if r else None

        # ----- Código -----
        codigo_val = row._1
        if codigo_val not in cache_codigo:
            r = conn.execute(
                text("SELECT id FROM icd10pcs WHERE codigo = :codigo"),
                {"codigo": codigo_val}
            ).fetchone()
            cache_codigo[codigo_val] = r[0] if r else None

        # ----- Preparar registro para inserção -----
        registros.append({
            "tabela_origem": "icd10pcs",
            "codigo_id": cache_codigo[codigo_val],
            "category_id": cache_category[nome_categoria],
            "specialty_id": specialty_id,
            "section_id": cache_section[nome_secao]
        })

# ==========================
# 5) Inserção em massa
# ==========================
with engine.connect() as conn:
    conn.execute(
        text("""
            INSERT INTO favoritos (tabela_origem, codigo_id, category_id, specialty_id, section_id)
            VALUES (:tabela_origem, :codigo_id, :category_id, :specialty_id, :section_id)
        """),
        registros
    )
    conn.commit()

print("Inserção concluída com sucesso!")



{'Cirurgia Geral': 10}
Inserção concluída com sucesso!
